In [23]:
import os
os.chdir(r"C:\Users\Lenovo\placement tracker")
print(os.getcwd())  # confirms you're now in the right folder
import requests
import time
import json
from datetime import datetime

URL = "https://placements25-26.vercel.app/api/placements"
OUTPUT_FILE = "placements_data.json"
LOG_FILE = "scrape_log.txt"
RETRY_INTERVAL_MINUTES = 15

def log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

def try_fetch():
    try:
        response = requests.get(URL, timeout=15)
        data = response.json()

        # Check if it's the same 429/error shape we've been seeing
        if isinstance(data, dict) and data.get("ok") is False:
            log(f"Still failing -> {data.get('error')}")
            return None

        # If we got here, looks like real data
        return data

    except requests.exceptions.RequestException as e:
        log(f"Request error: {e}")
        return None
    except json.JSONDecodeError:
        log("Response wasn't valid JSON")
        return None

log("Starting overnight polling. Will retry every 15 minutes until success.")

attempt = 1
while True:
    log(f"Attempt {attempt}: trying {URL}")
    result = try_fetch()

    if result is not None:
        log("SUCCESS! Got real data. Saving to file and stopping.")
        with open(OUTPUT_FILE, "w") as f:
            json.dump(result, f, indent=2)
        log(f"Saved to {OUTPUT_FILE}. You can stop the laptop now.")
        break

    attempt += 1
    log(f"Sleeping {RETRY_INTERVAL_MINUTES} minutes before next attempt...\n")
    time.sleep(RETRY_INTERVAL_MINUTES * 60)

C:\Users\Lenovo\placement tracker
[2026-08-30 15:10:14] Starting overnight polling. Will retry every 15 minutes until success.
[2026-08-30 15:10:14] Attempt 1: trying https://placements25-26.vercel.app/api/placements
[2026-08-30 15:10:16] SUCCESS! Got real data. Saving to file and stopping.
[2026-08-30 15:10:16] Saved to placements_data.json. You can stop the laptop now.


In [24]:
  import json

In [25]:
with open("placements_data.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

In [26]:
type(raw)

dict

In [27]:
raw.keys()

dict_keys(['ok', 'placements', 'branchStatsOverrides', 'fetchedAt', 'count'])

In [28]:
raw['ok']

True

In [29]:
raw['count']

538

In [30]:
raw['fetchedAt']

'2026-08-30T09:37:36.115Z'

In [31]:
type(raw['placements'][0]['offers'][0])
     

dict

In [32]:
type(raw['placements'][0]['offers'])

list

In [33]:
placements = raw['placements']
rows = []

In [34]:
import json
import re
import pandas as pd
 
# ---- Load the raw file ----
with open("placements_data.json", "r", encoding="utf-8") as f:
    raw = json.load(f)
 
print(f"Data fetched at: {raw['fetchedAt']}")
print(f"Total placement notices: {raw['count']}")
 
placements = raw["placements"]
 
# ---- Flatten: one row per offer, not per company ----
rows = []
for notice in placements:
    company = notice.get("companyName", "Unknown")
    notice_date = notice.get("noticeDate", "")
    for offer in notice.get("offers", []):
        rows.append({
            "company": company,
            "notice_date": notice_date,
            "job_role": offer.get("jobRole", ""),
            "job_type": offer.get("type", ""),
            "ctc_raw": offer.get("ctc", ""),
            "stipend_raw": offer.get("stipend", ""),
            "cgpa_raw": offer.get("eligibilityCgpa", ""),
            "eligibility_note_raw": offer.get("eligibilityNote", ""),
            "branches_allowed": offer.get("branchesAllowed", []),
            "students_selected": offer.get("studentsSelected", ""),
        })
 
df = pd.DataFrame(rows)
print(f"\nFlattened into {len(df)} individual offer rows.")
print(df.head())

Data fetched at: 2026-08-30T09:37:36.115Z
Total placement notices: 538

Flattened into 768 individual offer rows.
                                             company notice_date  \
0                       Pioneer E Solutions Pvt. Ltd  06/07/2026   
1                       Pioneer E Solutions Pvt. Ltd  06/07/2026   
2                       Pioneer E Solutions Pvt. Ltd  06/07/2026   
3                       Pioneer E Solutions Pvt. Ltd  06/07/2026   
4  Thapar Institute of Engineering & Technology P...  29/06/2026   

                                            job_role job_type         ctc_raw  \
0  Team Lead – Full Stack Developer(Pioneer on Be...      FTE  960000-1020000   
1  Assistant Advisor (Pioneer on behalf of Nation...      FTE  960000-1020000   
2  Team Lead – Full Stack Developer(Pioneer on Be...      FTE  960000-1020000   
3  Assistant Advisor (Pioneer on behalf of Nation...      FTE  960000-1020000   
4       Graduate Engineer Trainees (GETs) Electrical      FTE          5

In [35]:
df['cgpa_raw'].value_counts().reset_index()

,cgpa_raw,count
0,No CGPA Criteria,244
1,6,142
2,7,121
3,6.5,71
4,8,49
5,7.5,47
6,Not Applicable,20
7,8.5,16
8,N.A.,14
9,9,5


In [36]:
def parse_cgpa_range(value):
    if not value:
        return (None, None)
    numbers = [float(n) for n in re.findall(r"\d+\.?\d*", str(value))]
    if not numbers:
        return (None, None)          # e.g. "No CGPA Criteria", "Not Disclosed"
    if len(numbers) == 1:
        return (numbers[0], None)    # simple lower cutoff only, e.g. "7.5"
    return (numbers[0], numbers[1])  # range, e.g. "6.5 - 9.5" -> (6.5, 9.5)



In [37]:
def format_cgpa_display(row):
    """Single human-readable column: shows a range as 'min-max', a single
    cutoff as just the number, and blank when no cutoff applies."""
    lo, hi = row["cgpa_min"], row["cgpa_max"]
    if pd.isna(lo) and pd.isna(hi):
        return ""
    if pd.notna(hi):
        return f"{lo}-{hi}"
    return f"{lo}"
 
df[["cgpa_min", "cgpa_max"]] = df["cgpa_raw"].apply(
    lambda v: pd.Series(parse_cgpa_range(v))
)
df["cgpa_display"] = df.apply(format_cgpa_display, axis=1)

In [38]:
def parse_internal_cut(note):
    if not note or "internal cut" not in str(note).lower():
        return None
    match = re.search(r"\d+\.?\d*", str(note))
    return float(match.group()) if match else None
 
df["internal_cut"] = df["eligibility_note_raw"].apply(parse_internal_cut)
df["internal_cut_note"] = df["eligibility_note_raw"].where(
    df["internal_cut"].notna(), None
)

In [39]:
def format_cgpa_with_internal(row):
    base = row["cgpa_display"]
    if pd.notna(row["internal_cut"]):
        suffix = f" (Internal: {row['internal_cut']})"
        return (base + suffix) if base else f"No stated cutoff{suffix}"
    return base
 
df["cgpa_full_display"] = df.apply(format_cgpa_with_internal, axis=1)
df

,company,notice_date,job_role,job_type,ctc_raw,stipend_raw,cgpa_raw,eligibility_note_raw,branches_allowed,students_selected,cgpa_min,cgpa_max,cgpa_display,internal_cut,internal_cut_note,cgpa_full_display
0,Pioneer E Solutions Pvt. Ltd,06/07/2026,Team Lead – Full Stack Developer(Pioneer on Be...,FTE,960000-1020000,,No CGPA Criteria,,[M.E./MTech],Process Pending,NaN,NaN,,NaN,None,
1,Pioneer E Solutions Pvt. Ltd,06/07/2026,Assistant Advisor (Pioneer on behalf of Nation...,FTE,960000-1020000,,7.5,,"[COPC, COE, COBS, ENC, ECE, EIC, EEC, CIE]",Process Pending,7.5,NaN,7.5,NaN,None,7.5
2,Pioneer E Solutions Pvt. Ltd,06/07/2026,Team Lead – Full Stack Developer(Pioneer on Be...,FTE,960000-1020000,,No CGPA Criteria,,[M.E./MTech],Process Pending,NaN,NaN,,NaN,None,
3,Pioneer E Solutions Pvt. Ltd,06/07/2026,Assistant Advisor (Pioneer on behalf of Nation...,FTE,960000-1020000,,7.5,,"[COPC, COE, COBS, ENC, ECE, EIC, EEC, CIE]",Process Pending,7.5,NaN,7.5,NaN,None,7.5
4,Thapar Institute of Engineering & Technology P...,29/06/2026,Graduate Engineer Trainees (GETs) Electrical,FTE,500000,,6,,[ELE],Process Pending,6.0,NaN,6.0,NaN,None,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,NXP Semiconductors India,09/09/2025,"Digital,Analog Engineer and Embedded Software ...",Intern+Performance Based Chance of FTE,1550000,43000,8,Internal Cut for 2nd Written Round: 9.40,"[ENC, ECE, EIC]","4 (Only 1 FTE Conversion; 1 not given, 1 prior...",8.0,NaN,8.0,2.0,Internal Cut for 2nd Written Round: 9.40,8.0 (Internal: 2.0)
764,Barclays,14/08/2025,Not Known,PPO from Summer Intern,1350000,,N.A.,,[Not Applicable],19,NaN,NaN,,NaN,None,
765,Expedia Group,29/08/2025,Software Development Engineer I,PPO (Summer Intern/Competition),3200000,,Not Applicable,,[Not Applicable],5,NaN,NaN,,NaN,None,
766,Expedia Group,29/08/2025,Mobile Engineer I,PPO (Summer Intern/Competition),3200000,,Not Applicable,,[Not Applicable],3,NaN,NaN,,NaN,None,


In [40]:
def parse_ctc_range_lpa(value):
    if not value:
        return (None, None)
    numbers = [int(n) for n in re.findall(r"\d+", str(value))]
    if not numbers:
        return (None, None)
    lpa_values = [round(n / 100000, 2) for n in numbers]
    if len(lpa_values) == 1:
        return (lpa_values[0], lpa_values[0])
    return (min(lpa_values), max(lpa_values))
 
def format_ctc_display(row):
    lo, hi = row["ctc_min_lpa"], row["ctc_max_lpa"]
    if pd.isna(lo):
        return ""
    if lo == hi:
        return f"{lo} LPA"
    return f"{lo}-{hi} LPA"
 
df[["ctc_min_lpa", "ctc_max_lpa"]] = df["ctc_raw"].apply(
    lambda v: pd.Series(parse_ctc_range_lpa(v))
)
df["ctc_display"] = df.apply(format_ctc_display, axis=1)

In [41]:
def parse_students_selected(value):
    if value is None:
        return None
    text = str(value).strip()
    if text.isdigit():
        return int(text)
    return None  # not a plain number -- see selection_status for the detail
 
def selection_status_label(value):
    if value is None or str(value).strip() == "":
        return "Not known"
    text = str(value).strip()
    if text.isdigit():
        return f"{text} selected"
    return text  # keep the original status text as-is, e.g. "Process Pending"
 
df["students_selected_num"] = df["students_selected"].apply(parse_students_selected)
df["selection_status"] = df["students_selected"].apply(selection_status_label)

In [42]:
# ---- Clean dates: notice_date is a string like "06/07/2026" (DD/MM/YYYY) ----
df["notice_date"] = pd.to_datetime(df["notice_date"], format="%d/%m/%Y", errors="coerce")

In [43]:
df_by_branch = df.explode("branches_allowed").rename(columns={"branches_allowed": "branch"})
 
# ---- Save cleaned data for the next steps ----
df.to_csv("placements_clean.csv", index=False)
df_by_branch.to_csv("placements_by_branch.csv", index=False)
 
print("\nSaved placements_clean.csv and placements_by_branch.csv")
print(f"\nUnique branches found: {sorted(df_by_branch['branch'].dropna().unique())}")
print(f"CTC range (LPA): {df['ctc_min_lpa'].min()} to {df['ctc_max_lpa'].max()}")


Saved placements_clean.csv and placements_by_branch.csv

Unique branches found: ['B.E. All Branches', 'BME', 'BT', 'CHE', 'CIE', 'COBS', 'COE', 'COPC', 'ECE', 'EE', 'EEC', 'EIC', 'ELE', 'ENC', 'M.E.', 'M.E./MTech', 'M.Sc.', 'MCA', 'ME', 'MEC', 'MEE', 'Not Applicable', 'Not Known']
CTC range (LPA): 0.0 to 123.0
